In [5]:
import os
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

def convert_iccn_decree_to_md(pdf_path: str, output_md_path: str):
    # 1. Configure pipeline options to ensure strict table structure recognition
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_table_structure = True 
    
    # 2. Initialize the converter with the PDF options
    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    
    print(f"Parsing '{pdf_path}'... This might take a moment depending on your hardware.")
    
    # 3. Convert the document
    result = converter.convert(pdf_path)
    
    # 4. Export the result to standard Markdown
    markdown_text = result.document.export_to_markdown()
    
    # 5. Save to a file
    with open(output_md_path, "w", encoding="utf-8") as f:
        f.write(markdown_text)
        
    print(f"Success! Clean Markdown saved to '{output_md_path}'.")

if __name__ == "__main__":
    # Ensure the file name exactly matches your local file
    input_pdf = "SK36_PENGURUS_ICCN.pdf"
    output_markdown = "Parsed.md"
    
    if os.path.exists(input_pdf):
        convert_iccn_decree_to_md(input_pdf, output_markdown)
    else:
        print(f"Error: Could not find the file {input_pdf} in the current directory.")

[INFO] 2026-03-22 14:48:56,898 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-22 14:48:56,910 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-22 14:48:56,911 [RapidOCR] main.py:53: Using C:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx


Parsing 'SK36_PENGURUS_ICCN.pdf'... This might take a moment depending on your hardware.


[INFO] 2026-03-22 14:48:57,191 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-22 14:48:57,191 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-22 14:48:57,191 [RapidOCR] main.py:53: Using C:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-22 14:48:57,321 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-22 14:48:57,342 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
[INFO] 2026-03-22 14:48:57,346 [RapidOCR] main.py:53: Using C:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx


Success! Clean Markdown saved to 'Parsed.md'.


In [3]:
"""
AskDoc - PDF to Markdown Preprocessor (POC)
============================================
Uses Docling to parse PDFs (including table-heavy docs like SK/susunan pengurus)
into clean Markdown, ready to be fed into PageIndex or any RAG pipeline.

Install dependencies:
    pip install docling pymupdf

Usage:
    python parse_pdf_docling.py --input SK36_Pengurus_ICCN_2025-2028_.pdf
"""

import argparse
import json
import re
import sys
from pathlib import Path


# ─────────────────────────────────────────────
# STEP 1: Detect if PDF is scanned or digital
# ─────────────────────────────────────────────

def is_scanned_pdf(pdf_path: str, threshold: int = 100) -> bool:
    """
    Uses PyMuPDF to check if a PDF has extractable text.
    If total extracted text is below threshold → treat as scanned.
    """
    try:
        import fitz  # PyMuPDF
        doc = fitz.open(pdf_path)
        total_text = "".join(page.get_text() for page in doc)
        doc.close()
        is_scanned = len(total_text.strip()) < threshold
        print(f"[detect] Extracted {len(total_text.strip())} chars → {'SCANNED' if is_scanned else 'DIGITAL'} PDF")
        return is_scanned
    except ImportError:
        print("[warn] PyMuPDF not installed, assuming digital PDF")
        return False


# ─────────────────────────────────────────────
# STEP 2: Parse with Docling (primary parser)
# ─────────────────────────────────────────────

def parse_with_docling(pdf_path: str, enable_ocr: bool = False) -> dict:
    """
    Convert PDF to Markdown using Docling.
    Returns dict with 'markdown', 'tables', and 'metadata'.

    For scanned PDFs, enable_ocr=True activates Docling's OCR pipeline.
    """
    from docling.document_converter import DocumentConverter, PdfFormatOption
    from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
    from docling.datamodel.base_models import InputFormat

    print(f"[docling] Parsing: {pdf_path} (OCR={'on' if enable_ocr else 'off'})")

    # Configure pipeline
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = enable_ocr
    pipeline_options.do_table_structure = True  # Enable TableFormer model
    pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE  # vs FAST

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    result = converter.convert(pdf_path)
    doc = result.document

    # Export to Markdown
    markdown = doc.export_to_markdown()

    # Extract tables separately as structured JSON (compatible with Docling v2+)
    tables = []
    for i, table in enumerate(doc.tables):
        try:
            table_md = table.export_to_markdown()
        except Exception:
            table_md = ""

        try:
            table_df = table.export_to_dataframe()
            num_rows = len(table_df)
            num_cols = len(table_df.columns)
            records = table_df.to_dict(orient="records")
        except Exception:
            # Fallback: get dimensions from table.data.grid if available
            grid = getattr(getattr(table, "data", None), "grid", None)
            num_rows = len(grid) if grid else 0
            num_cols = len(grid[0]) if grid else 0
            records = []

        tables.append({
            "index": i,
            "num_rows": num_rows,
            "num_cols": num_cols,
            "markdown": table_md,
            "dataframe": records
        })

    metadata = {
        "source": pdf_path,
        "num_pages": len(doc.pages),
        "num_tables": len(tables),
        "ocr_used": enable_ocr,
    }

    print(f"[docling] Done: {metadata['num_pages']} pages, {metadata['num_tables']} tables found")

    return {
        "markdown": markdown,
        "tables": tables,
        "metadata": metadata
    }


# ─────────────────────────────────────────────
# STEP 3: Post-process Markdown
# ─────────────────────────────────────────────

def postprocess_markdown(raw_markdown: str) -> str:
    """
    Clean up the Docling Markdown output:
    - Normalize whitespace
    - Ensure tables are properly formatted
    - Remove artifacts (page headers/footers repeated across pages)
    """

    lines = raw_markdown.split("\n")
    cleaned_lines = []
    seen_repeats = {}

    for line in lines:
        stripped = line.strip()

        # Skip common repeated header/footer lines (e.g., org name, email)
        if stripped in [
            "Indonesia Creative Cities Network (ICCN)",
            "creativecitiesid@gmail.com | @iccnmedia",
            ""
        ]:
            # Allow blank lines but skip repeated headers
            if stripped == "":
                cleaned_lines.append("")
            continue

        # Track repeated lines that appear 3+ times (likely headers/footers)
        seen_repeats[stripped] = seen_repeats.get(stripped, 0) + 1
        if seen_repeats[stripped] > 3 and not stripped.startswith("|"):
            continue

        cleaned_lines.append(line)

    # Collapse multiple consecutive blank lines into one
    result = re.sub(r'\n{3,}', '\n\n', "\n".join(cleaned_lines))
    return result.strip()


# ─────────────────────────────────────────────
# STEP 4: Save outputs
# ─────────────────────────────────────────────

def save_outputs(result: dict, output_dir: str = "output"):
    """
    Save:
    - clean_output.md   → full clean Markdown (ready for PageIndex)
    - tables.json       → extracted tables as structured JSON
    - metadata.json     → document metadata
    """
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)

    # Clean markdown
    clean_md = postprocess_markdown(result["markdown"])
    md_path = out / "clean_output.md"
    md_path.write_text(clean_md, encoding="utf-8")
    print(f"[save] Markdown → {md_path}")

    # Tables JSON
    tables_path = out / "tables.json"
    tables_path.write_text(json.dumps(result["tables"], indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[save] Tables   → {tables_path} ({len(result['tables'])} tables)")

    # Metadata
    meta_path = out / "metadata.json"
    meta_path.write_text(json.dumps(result["metadata"], indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"[save] Metadata → {meta_path}")

    return str(md_path)


# ─────────────────────────────────────────────
# STEP 5: Feed into PageIndex (stub)
# ─────────────────────────────────────────────

def index_with_pageindex(markdown_path: str):
    """
    Stub showing how to pass clean Markdown into PageIndex.
    Replace with your actual PageIndex integration.
    """
    print(f"\n[pageindex] Indexing: {markdown_path}")
    print("[pageindex] Note: PageIndex receives clean Markdown with proper table syntax")

    # Example integration (uncomment when PageIndex is available):
    # from pageindex import PageIndex
    # index = PageIndex(markdown_path)
    # response = index.query("Siapa Ketua Umum ICCN?")
    # print(response)

    # For now, just show a preview
    content = Path(markdown_path).read_text(encoding="utf-8")
    print(f"\n--- Markdown Preview (first 1000 chars) ---\n")
    print(content[:1000])
    print("...\n")


# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────

def run_pipeline(pdf_path: str, output_dir: str = "output", force_ocr: bool = False):
    """
    Full pipeline:
    PDF → detect → Docling parse → clean Markdown → PageIndex
    """
    print(f"\n{'='*50}")
    print(f"AskDoc PDF Preprocessor")
    print(f"Input: {pdf_path}")
    print(f"{'='*50}\n")

    # 1. Detect doc type
    scanned = is_scanned_pdf(pdf_path) if not force_ocr else True

    # 2. Parse
    result = parse_with_docling(pdf_path, enable_ocr=scanned or force_ocr)

    # 3. Save outputs
    md_path = save_outputs(result, output_dir)

    # 4. Index (stub)
    index_with_pageindex(md_path)

    print(f"\n✅ Pipeline complete. Clean Markdown at: {md_path}")
    return md_path

run_pipeline(
    pdf_path="SK36_PENGURUS_ICCN.pdf",
    output_dir="output",
    force_ocr=False  # ← explicit
)


AskDoc PDF Preprocessor
Input: SK36_PENGURUS_ICCN.pdf

[detect] Extracted 15189 chars → DIGITAL PDF
[docling] Parsing: SK36_PENGURUS_ICCN.pdf (OCR=off)


Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_markdown() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


[docling] Done: 9 pages, 6 tables found
[save] Markdown → output\clean_output.md
[save] Tables   → output\tables.json (6 tables)
[save] Metadata → output\metadata.json

[pageindex] Indexing: output\clean_output.md
[pageindex] Note: PageIndex receives clean Markdown with proper table syntax

--- Markdown Preview (first 1000 chars) ---

## SURAT KEPUTUSAN

## DEWAN PENGURUS PERKUMPULAN JEJARING KOTA DAN KABUPATEN KREATIF

## DI INDONESIA Nomor: 36/SK-ICCN/XII/2025

## Tentang:

## SUSUNAN PENGURUS PERKUMPULAN JEJARING KOTA DAN KABUPATEN KREATIF DI INDONESIA

## PERIODE TAHUN 2025-2028

DEWAN PENGURUS PERKUMPULAN JEJARING KOTA DAN KABUPATEN KREATIF DI INDONESIA :

## Menimbang :

<!-- image -->

1. Bahwa untuk memperlancar pencapaian Visi dan Misi perkumpulan sebagaimana tertuang dalam Anggaran Dasar Perkumpulan, maka diperlukan organisasi yang solid.
2. Pentingnya  susunan  Pengurus  Perkumpulan  untuk  memperlancar Pelaksanaan  kegiatan  organisasi  Perkumpulan  Jejaring  Kota  dan Kabu

'output\\clean_output.md'

c:\Users\satyananda\Documents\AskDocs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: <module 'src.pageindex.indexer.page_index_md' from 'c:\\Users\\satyananda\\Documents\\AskDocs\\src\\pageindex\\indexer\\page_index_md.py'> does not have the attribute 'OpenAI'